In [1]:
import json
import os
from datetime import datetime
import numpy as np

In [2]:
def as_list_on_duplicate_keys(ordered_pairs):
    """
    A custom JSON object_pairs_hook that collects values for duplicate
    keys into a list. Essential for handling Wireshark JSONs.
    """
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list):
                d[k].append(v)
            else:
                d[k] = [d[k], v]
        else:
            d[k] = v
    return d

In [3]:
def extract_simplified_features(json_file_path, encoding='utf-8'):
    """
    Analyzes a decrypted QUIC packet capture from a Wireshark JSON export
    and extracts a simplified, high-level feature set for a baseline model.

    Args:
        json_file_path (str): The path to the JSON file.
        encoding (str): The file encoding to use.

    Returns:
        dict: A dictionary containing the extracted features for the flow.
              Returns None if the capture is empty or invalid.
    """
    with open(json_file_path, 'r', encoding=encoding) as f:
        packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)

    if not packets:
        print(f"Capture file '{json_file_path}' is empty.")
        return None

    features = {}
    first_packet = packets[0]['_source']['layers']
    last_packet = packets[-1]['_source']['layers']

    initial_ip_client = first_packet['ip']['ip.src']
    initial_ip_server = first_packet['ip']['ip.dst']
    initial_port_client = int(first_packet['udp']['udp.srcport'])
    
    time_first_epoch = float(first_packet['frame']['frame.time_epoch'])
    time_last_epoch = float(last_packet['frame']['frame.time_epoch'])
    features['connection_duration_msec'] = (time_last_epoch - time_first_epoch) * 1000

    handshake_start_time = None
    handshake_end_time = None
    features['retry_occurred'] = 0
    
    client_bidi_stream_ids = set()
    client_uni_stream_ids = set()
    server_uni_stream_ids = set() 
    
    total_client_app_bytes = 0
    total_server_app_bytes = 0
    client_app_data_chunks = []
    server_app_data_chunks = []

    migrated = False
    migrated_ip_client = None
    migration_start_time = None
    path_challenge_time = None
    path_response_time = None
    app_data_bytes_before_migration = 0
    features['server_issued_cid_count'] = 0

    # --- 2. Main Packet Iteration Loop ---
    for i, pkt_data in enumerate(packets):
        layers = pkt_data['_source']['layers']
        
        current_time = float(layers['frame']['frame.time_epoch'])
        src_ip = layers['ip']['ip.src']
        dst_ip = layers['ip']['ip.dst']
        src_port = int(layers['udp']['udp.srcport'])
        
        # --- Migration Detection ---
        if not migrated and (dst_ip == initial_ip_server) and \
           (src_ip != initial_ip_client or src_port != initial_port_client):
            
            migrated = True
            migration_start_time = current_time
            migrated_ip_client = src_ip
            
            ip_changed = src_ip != initial_ip_client
            port_changed = src_port != initial_port_client
            if ip_changed and port_changed:
                features['migration_type'] = 'IP_AND_PORT'
            elif ip_changed:
                features['migration_type'] = 'IP_ONLY'
            elif port_changed:
                features['migration_type'] = 'PORT_ONLY'

        is_client_pkt = (src_ip == initial_ip_client or src_ip == migrated_ip_client)

        # There can be multiple QUIC packets in a single UDP datagram
        quic_packet_list = layers.get('quic', [])
        if not isinstance(quic_packet_list, list):
            quic_packet_list = [quic_packet_list]

        for quic_packet in quic_packet_list:
            if quic_packet.get('quic.long.packet_type') == '3': # Retry packet
                features['retry_occurred'] = 1
            if quic_packet.get('quic.long.packet_type') == '0': # Initial packet
                if handshake_start_time is None:
                    handshake_start_time = current_time

            quic_frames = quic_packet.get('quic.frame', [])
            if not isinstance(quic_frames, list):
                quic_frames = [quic_frames]
                
            for frame in quic_frames:
                frame_type = frame.get('quic.frame_type')
                if not frame_type: continue

                # Handshake timing
                if frame_type == '0x000000000000001e': # HANDSHAKE_DONE frame
                    if handshake_end_time is None:
                        handshake_end_time = current_time
                
                # Server readiness for migration
                if frame_type == '0x0000000000000018' and not is_client_pkt: # NEW_CONNECTION_ID
                    features['server_issued_cid_count'] += 1

                # Migration validation timing
                if frame_type == '0x000000000000001a' and is_client_pkt: # PATH_CHALLENGE
                    path_challenge_time = current_time
                if frame_type == '0x000000000000001b' and not is_client_pkt: # PATH_RESPONSE
                    if path_response_time is None: # Only capture the first response
                         path_response_time = current_time

                # Stream and Application Byte Analysis
                if '0x0000000000000008' <= frame_type <= '0x000000000000000f': # Any STREAM frame
                    stream_id = int(frame.get('quic.stream.stream_id', 0))
                    stream_len = int(frame.get('quic.stream.length', 0))
                    
                    # Count stream types
                    if stream_id % 4 == 0: client_bidi_stream_ids.add(stream_id)
                    elif stream_id % 4 == 2: client_uni_stream_ids.add(stream_id)
                    elif stream_id % 4 == 3: server_uni_stream_ids.add(stream_id)

                    # Accumulate bytes and data chunks
                    if is_client_pkt:
                        total_client_app_bytes += stream_len
                        client_app_data_chunks.append(stream_len)
                    else:
                        total_server_app_bytes += stream_len
                        server_app_data_chunks.append(stream_len)
                    
                    if not migrated:
                        app_data_bytes_before_migration += stream_len

                # Connection Close
                if frame_type in ('0x000000000000001c', '0x000000000000001d'):
                    features['connection_close_type'] = 'CLIENT_CLOSE' if is_client_pkt else 'SERVER_CLOSE'

    # --- 3. Final Calculations and Feature Assembly ---
    # Durations
    if handshake_start_time and handshake_end_time:
        features['handshake_duration_msec'] = (handshake_end_time - handshake_start_time) * 1000
    else:
        features['handshake_duration_msec'] = None

    # App Bytes and Averages
    features['total_client_app_bytes'] = total_client_app_bytes
    features['total_server_app_bytes'] = total_server_app_bytes
    features['avg_request_size'] = np.mean(client_app_data_chunks) if client_app_data_chunks else 0
    features['avg_response_size'] = np.mean(server_app_data_chunks) if server_app_data_chunks else 0
    
    # Stream Counts
    features['client_bidi_streams_count'] = len(client_bidi_stream_ids)
    features['client_uni_streams_count'] = len(client_uni_stream_ids)
    features['server_uni_streams_count'] = len(server_uni_stream_ids)
    
    # Migration Features
    if migrated:
        features['time_to_migration_msec'] = (migration_start_time - time_first_epoch) * 1000
        features['app_data_bytes_before_migration'] = app_data_bytes_before_migration
        if path_challenge_time and path_response_time:
            features['migration_validation_duration_msec'] = (path_response_time - path_challenge_time) * 1000
        else:
            features['migration_validation_duration_msec'] = None
    else:
        # Set default values for non-migrating flows
        features['migration_type'] = 'NO_MIGRATION'
        features['time_to_migration_msec'] = 0
        features['app_data_bytes_before_migration'] = 0
        features['migration_validation_duration_msec'] = 0

    # Final check for connection close type
    if 'connection_close_type' not in features:
        features['connection_close_type'] = 'IDLE_TIMEOUT'
        
    return features

In [4]:
def process_quic_capture_wrapper(json_file_path):
    """
    Wrapper function that determines the correct encoding (UTF-8 or UTF-16) 
    for a JSON file and then calls the feature extraction function.
    """
    if not os.path.exists(json_file_path) or os.path.getsize(json_file_path) == 0:
        print(f"Skipping '{json_file_path}': File is empty or does not exist.")
        return None

    try:
        return extract_simplified_features(json_file_path, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            return extract_simplified_features(json_file_path, encoding='utf-16')
        except Exception as e:
            print(f"ERROR: Failed to process '{os.path.basename(json_file_path)}' as UTF-16. Error: {e}")
            return None
    except Exception as e:
        print(f"ERROR: Failed to process '{os.path.basename(json_file_path)}'. Error: {e}")
        return None

In [5]:
capture_file = 'C:/Users/vassa/Desktop/UZH/Masters Project/synthetic_network_data_gen/captures_json/quiche/quiche_capture_1.json'

if os.path.exists(capture_file):
    features = process_quic_capture_wrapper(capture_file)
    if features:
        print("Successfully extracted features:")
        for key, value in features.items():
            print(f"  - {key}: {value}")
else:
    print(f"Example file not found: '{capture_file}'")
    print("Please update the 'capture_file' variable with the path to your JSON export.")

Successfully extracted features:
  - connection_duration_msec: 107.25188255310059
  - retry_occurred: 1
  - server_issued_cid_count: 1
  - migration_type: IP_AND_PORT
  - connection_close_type: CLIENT_CLOSE
  - handshake_duration_msec: 77.66485214233398
  - total_client_app_bytes: 119
  - total_server_app_bytes: 355
  - avg_request_size: 23.8
  - avg_response_size: 71.0
  - client_bidi_streams_count: 1
  - client_uni_streams_count: 4
  - server_uni_streams_count: 4
  - time_to_migration_msec: 86.83586120605469
  - app_data_bytes_before_migration: 47
  - migration_validation_duration_msec: 1.1742115020751953
